# Take-home 3 giờ: cắt 20% mã hàng năm 2012?

Đề đầy đủ ở `README.md` của bài. Tóm tắt: Head of Merchandising muốn cắt khoảng 20% số mã hàng (SKU) trong năm 2012 để
giảm tồn kho, và hỏi **nên cắt mã nào, rủi ro là gì**. Dữ liệu: Online Retail II, sheet `Year 2010-2011`.

**Bấm giờ ngay bây giờ.** Ghi giờ bắt đầu và kết thúc từng phần vào mục `## Nhật ký thời gian` của `SUBMISSION.md`.

**Cách làm bài**
- Ba hàm được chấm trên dữ liệu nhỏ: `clean_product_lines`, `sku_table`, `cut_impact`. Phần còn lại là phân tích của bạn
  trong các cell `explore` (không chấm tự động, chấm theo rubric).
- Sản phẩm nộp: notebook này chạy được từ đầu tới cuối + `SUBMISSION.md`.
- Chạy test: `uv run pytest tracks/data-analyst/modules/15-stakeholder-storytelling-interview/exercises/advanced/01-take-home-3h -q`.

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu, đọc sheet 2010-2011 (lần đầu 20-60 giây, sau đó đọc cache parquet)
import os
import sys
import tempfile
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd


def read_sheet(sheet):
    src = ensure("online-retail-ii") / "online_retail_II.xlsx"
    st = src.stat()
    cache = Path(tempfile.gettempdir()) / "da-cache" / f"online_retail_II-{sheet.replace(' ', '_')}-{st.st_size}-{st.st_mtime_ns}.parquet"
    if not cache.exists():
        df = pd.read_excel(src, sheet_name=sheet, dtype={"Invoice": str, "StockCode": str, "Description": str})
        cache.parent.mkdir(parents=True, exist_ok=True)
        tmp = cache.with_name(f"{cache.name}.{os.getpid()}.tmp")
        df.to_parquet(tmp, index=False)
        os.replace(tmp, cache)
    return pd.read_parquet(cache)


tx = read_sheet("Year 2010-2011")
print(tx.shape, tx["InvoiceDate"].min(), "->", tx["InvoiceDate"].max())

In [ ]:
import pandas as pd

# Mã hàng hoá: 5 chữ số, có thể kèm chữ cái (85123A, 15056BL). POST, DOT, M, AMAZONFEE, gift_0001_20... không phải hàng.
PRODUCT_CODE = r"^\d{5}[A-Z]*$"

## 1. Dòng sản phẩm sạch

`clean_product_lines(tx) -> DataFrame`: bản sao chỉ gồm dòng hợp lệ theo định nghĩa dùng chung của track (bỏ `Invoice` bắt
đầu bằng `C`, bỏ `Quantity <= 0` hoặc `Price <= 0`) **và** là hàng hoá: `StockCode` sau khi ép `str`, bỏ khoảng trắng hai
đầu và viết hoa khớp `PRODUCT_CODE`. Trả `StockCode` đã chuẩn hoá, `Invoice` là `str`, thêm cột `revenue = Quantity * Price`,
giữ index gốc. Kỳ phân tích (12 tháng nào) là quyết định của bạn, **không** lọc trong hàm này.

In [ ]:
def clean_product_lines(tx: pd.DataFrame) -> pd.DataFrame:
    """Dòng hợp lệ, chỉ hàng hoá, StockCode chuẩn hoá, có cột revenue."""
    # TODO
    raise NotImplementedError

## 2. Bảng mã hàng và tác động của một phương án cắt

`sku_table(sales) -> DataFrame`: index `StockCode`, cột `revenue` (tổng `Quantity * Price`), `units` (tổng `Quantity`),
`invoices` (số hoá đơn khác nhau), `customers` (số `Customer ID` khác nhau, không tính thiếu). Sắp `revenue` giảm dần,
hoà thì `StockCode` tăng dần.

`cut_impact(sales, cut_skus) -> dict` cho một danh sách mã định cắt:

| Khoá | Định nghĩa |
| --- | --- |
| `skus_cut` | Số mã trong `cut_skus` có xuất hiện trong `sales` (`int`) |
| `revenue_lost_share` | Doanh thu của các mã bị cắt / tổng doanh thu |
| `customers_affected` | Số khách (có mã) đã mua ít nhất một mã bị cắt (`int`) |
| `customers_affected_share` | `customers_affected` / số khách có mã |
| `affected_customers_revenue_share` | **Toàn bộ** doanh thu của các khách đó (mọi mã) / doanh thu của khách có mã |

Hai hàm không lọc lại dữ liệu và không sửa đầu vào. Khoá cuối là câu hỏi rủi ro: cắt mã của ai?

In [ ]:
def sku_table(sales: pd.DataFrame) -> pd.DataFrame:
    """Doanh thu, số lượng, số hoá đơn, số khách theo mã hàng."""
    # TODO
    raise NotImplementedError


def cut_impact(sales: pd.DataFrame, cut_skus) -> dict:
    """Doanh thu mất trực tiếp và khách bị ảnh hưởng khi cắt cut_skus."""
    # TODO
    raise NotImplementedError

## 3. Phân tích của bạn

Từ đây là của bạn. Gợi ý thứ tự (đừng làm hết nếu hết giờ: một câu trả lời rõ tốt hơn năm câu dở dang):

1. Chọn kỳ phân tích và ghi lý do (tháng 12/2011 chưa đủ; mã mới ra mắt cuối năm có ít doanh thu vì **mới**, không phải
   vì bán chậm).
2. Phương án ngây thơ: 20% mã doanh thu thấp nhất. Đo bằng `cut_impact`.
3. Ít nhất một phương án tốt hơn, có tiêu chí bạn giải thích được, và so hai phương án trên cùng các chỉ số.
4. Một chart cho người ra quyết định.

In [ ]:
# explore: phân tích của bạn